# Multi-Index 하이브리드 RAG 파이프라인

**Skilljar Lesson L07 대응**

이 노트북에서 다루는 내용:
1. Reciprocal Rank Fusion (RRF) 알고리즘
2. Retriever 클래스 구현 (VectorIndex + BM25Index 결합)
3. 하이브리드 검색 vs 단일 검색 비교
4. Claude와 연동한 Multi-Index RAG 질의응답

In [ ]:
# ── Setup ──────────────────────────────────────────────
import math
from collections import Counter

import anthropic
import chromadb
import voyageai
import numpy as np
from dotenv import load_dotenv

load_dotenv()

claude_client = anthropic.Anthropic()
voyage_client = voyageai.Client()
MODEL = "claude-haiku-4-5"

## §1. VectorIndex와 BM25Index (이전 노트북에서 가져옴)

In [ ]:
class VectorIndex:
    """ChromaDB 기반 벡터 검색 인덱스"""

    def __init__(self, collection_name="documents", embedding_model="voyage-3"):
        self.voyage_client = voyageai.Client()
        self.embedding_model = embedding_model
        self.chroma_client = chromadb.Client()
        self.collection = self.chroma_client.create_collection(
            name=collection_name, metadata={"hnsw:space": "cosine"}
        )

    def add_documents(self, documents, ids=None):
        if ids is None:
            ids = [f"doc_{i}" for i in range(len(documents))]
        result = self.voyage_client.embed(
            texts=documents, model=self.embedding_model, input_type="document"
        )
        self.collection.add(
            documents=documents, embeddings=result.embeddings, ids=ids
        )
        print(f"\u2705 VectorIndex: {len(documents)}개 문서 인덱싱 완료")

    def search(self, query, top_k=5):
        qr = self.voyage_client.embed(
            texts=[query], model=self.embedding_model, input_type="query"
        )
        results = self.collection.query(
            query_embeddings=qr.embeddings, n_results=top_k
        )
        return [
            {"text": results["documents"][0][i],
             "score": 1 - results["distances"][0][i],
             "id": results["ids"][0][i]}
            for i in range(len(results["documents"][0]))
        ]


class BM25Index:
    """BM25 기반 어휘 검색 인덱스"""

    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.documents, self.doc_lengths = [], []
        self.avg_doc_length = 0
        self.doc_freqs, self.doc_term_freqs = Counter(), []

    def _tokenize(self, text):
        return text.lower().split()

    def add_documents(self, documents):
        self.documents = documents
        for doc in documents:
            tokens = self._tokenize(doc)
            self.doc_lengths.append(len(tokens))
            tf = Counter(tokens)
            self.doc_term_freqs.append(tf)
            for t in set(tokens):
                self.doc_freqs[t] += 1
        self.avg_doc_length = sum(self.doc_lengths) / len(self.doc_lengths)
        print(f"\u2705 BM25Index: {len(documents)}개 문서 인덱싱 완료")

    def _idf(self, term):
        n = len(self.documents)
        df = self.doc_freqs.get(term, 0)
        return math.log((n - df + 0.5) / (df + 0.5) + 1)

    def search(self, query, top_k=5):
        tokens = self._tokenize(query)
        scores = []
        for i in range(len(self.documents)):
            score = 0
            for t in tokens:
                tf = self.doc_term_freqs[i].get(t, 0)
                idf = self._idf(t)
                dl = self.doc_lengths[i]
                num = tf * (self.k1 + 1)
                den = tf + self.k1 * (1 - self.b + self.b * dl / self.avg_doc_length)
                score += idf * num / den
            scores.append(score)
        top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        return [{"text": self.documents[i], "score": scores[i], "id": f"doc_{i}"} for i in top_idx]

## §2. Reciprocal Rank Fusion (RRF)

여러 검색 결과를 **순위(rank) 기반**으로 통합하는 알고리즘입니다.  
점수 스케일이 다른 두 검색을 정규화 없이 결합할 수 있습니다.

$$RRF\_score(d) = \sum_i \frac{1}{k + rank_i(d)}$$

여기서 $k$는 상수 (보통 60)입니다.

## §3. Retriever 클래스 구현

In [ ]:
class Retriever:
    """하이브리드 검색 Retriever — VectorIndex + BM25Index + RRF"""

    def __init__(self, vector_index: VectorIndex, bm25_index: BM25Index):
        self.vector_index = vector_index
        self.bm25_index = bm25_index

    def reciprocal_rank_fusion(
        self,
        semantic_results: list[dict],
        bm25_results: list[dict],
        k: int = 60
    ) -> list[dict]:
        """RRF로 두 검색 결과를 통합한다."""
        rrf_scores = {}

        for rank, result in enumerate(semantic_results, 1):
            doc_id = result["id"]
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank)

        for rank, result in enumerate(bm25_results, 1):
            doc_id = result["id"]
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank)

        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        all_results = {}
        for r in semantic_results + bm25_results:
            all_results[r["id"]] = r["text"]

        return [
            {"id": doc_id, "score": score, "text": all_results.get(doc_id, "")}
            for doc_id, score in sorted_docs
        ]

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        """하이브리드 검색을 수행한다."""
        semantic_results = self.vector_index.search(query, top_k=top_k * 2)
        bm25_results = self.bm25_index.search(query, top_k=top_k * 2)

        fused = self.reciprocal_rank_fusion(semantic_results, bm25_results)
        return fused[:top_k]

## §4. 인덱스 구축 (양쪽 모두)

In [ ]:
# 샘플 청크
chunks = [
    "KDS 41 10 20: 이 기준은 건축물의 구조안전성을 확보하기 위한 최소한의 요구사항을 정한다.",
    "고정하중은 구조물 자체의 무게와 영구적으로 부착된 부분의 무게를 포함한다. 콘크리트의 단위중량은 24 kN/m3이다.",
    "콘크리트의 설계기준강도 fck는 최소 21 MPa 이상이어야 한다. 고강도 콘크리트의 경우 fck 40 MPa 이상을 적용할 수 있다.",
    "RC 보의 최소 철근비는 0.25*sqrt(fck)/fy 이상이어야 하며, 1.4/fy 이상이어야 한다.",
    "기둥의 최소 단면치수는 300mm 이상이어야 한다. 주근의 최소 개수는 4개이다.",
    "적재하중은 건축물의 용도에 따라 다르게 적용한다. 주거용 2.0 kN/m2, 사무실 2.5 kN/m2이다.",
]

chunk_ids = [f"doc_{i}" for i in range(len(chunks))]

# 벡터 인덱스
vector_index = VectorIndex(collection_name="hybrid_test")
vector_index.add_documents(chunks, ids=chunk_ids)

# BM25 인덱스
bm25_index = BM25Index()
bm25_index.add_documents(chunks)

# Retriever 생성
retriever = Retriever(vector_index=vector_index, bm25_index=bm25_index)

## §5. 하이브리드 검색 vs 단일 검색 비교

In [ ]:
queries = [
    "KDS 41 10 20 구조 기준",     # 조항 번호 + 의미
    "RC 보의 최소 철근비",         # 의미적 검색
    "fck 40 MPa",               # 정확한 기술 용어
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"쿼리: '{query}'\n")

    # Semantic Only
    sem_results = vector_index.search(query, top_k=3)
    print("  [Semantic Only]")
    for i, r in enumerate(sem_results, 1):
        print(f"    {i}위 [{r['score']:.4f}] {r['text'][:60]}...")

    # BM25 Only
    bm25_results = bm25_index.search(query, top_k=3)
    print("  [BM25 Only]")
    for i, r in enumerate(bm25_results, 1):
        print(f"    {i}위 [{r['score']:.4f}] {r['text'][:60]}...")

    # Hybrid (RRF)
    hybrid_results = retriever.search(query, top_k=3)
    print("  [Hybrid RRF]")
    for i, r in enumerate(hybrid_results, 1):
        print(f"    {i}위 [RRF:{r['score']:.4f}] {r['text'][:60]}...")

## §6. Claude와 연동한 Multi-Index RAG

In [ ]:
def hybrid_rag_query(question: str, retriever: Retriever, top_k: int = 3) -> str:
    """하이브리드 RAG 질의응답"""
    results = retriever.search(question, top_k=top_k)
    context = "\n\n---\n\n".join([r["text"] for r in results])

    print(f"검색된 청크 {len(results)}개:")
    for r in results:
        print(f"  [RRF:{r['score']:.4f}] {r['text'][:60]}...")
    print()

    response = claude_client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=(
            "다음 문서를 참고하여 질문에 답하세요. "
            "문서에 없는 정보는 '문서에서 확인할 수 없습니다'라고 답하세요.\n\n"
            f"참고 문서:\n{context}"
        ),
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text

In [ ]:
# 하이브리드 RAG 질의응답
answer = hybrid_rag_query(
    "KDS 41 10 20에서 RC 보의 최소 철근비 기준을 설명해줘",
    retriever
)
print(f"답변:\n{answer}")

## 정리

- **Reciprocal Rank Fusion (RRF)**: 순위 기반 점수 통합, 정규화 불필요
- **Retriever 클래스**: VectorIndex + BM25Index를 RRF로 결합
- 하이브리드 검색은 Semantic과 Lexical의 **장점을 모두 활용**
- 조항 번호 + 의미적 검색이 동시에 필요한 건축공학에 특히 유용

학생 실습 노트북: `S4_06_rag_practice.ipynb`, `S4_07_structural_rag.ipynb`